In [ ]:
# !git clone https://github.com/TheFaridKarami/farid-ai-lab.git

In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  DataCollatorWithPadding,
  Trainer,
  TrainingArguments,
  pipeline,
  set_seed,
)

SEED = 42
set_seed(SEED)

In [ ]:
PROJECT_ROOT = Path(r"/Workspace/Projects/NLP/banking77-intent-classification")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [ ]:
print(PROJECT_ROOT.exists())
print(DATA_DIR.exists())
print(TRAIN_PATH.exists())
print(TEST_PATH.exists())

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Shapes:", train_df.shape, test_df.shape)
print(train_df.dtypes)
print("Nulls:\n", train_df.isnull().sum())
print(train_df.head())

counts = train_df["label"].value_counts()
print("Classes train/test:", train_df["label"].nunique(), test_df["label"].nunique())
print(counts.describe())
print(counts.tail(5))
print("Missing from test:", set(train_df["label"]) - set(test_df["label"]))

print("Duplicate rows:", train_df.duplicated().sum())
print("Duplicate texts train/test:", train_df["text"].duplicated().sum(), test_df["text"].duplicated().sum())
print("Train/test text overlap:", len(set(train_df["text"]) & set(test_df["text"])))
print("Texts with conflicting labels:", (train_df.groupby("text")["label"].nunique() > 1).sum())

In [ ]:
train_df = pd.read_csv(TRAIN_PATH).drop_duplicates(subset="text").reset_index(drop=True)
test_df  = pd.read_csv(TEST_PATH)
assert len(train_df) == 10003, len(train_df)

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    train_df["text"], train_df["label"],
    test_size=0.2, random_state=SEED, stratify=train_df["label"])

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(test_df["label"])

NUM_LABELS = len(le.classes_)
label2id = {str(l): int(i) for i, l in enumerate(le.classes_)}
id2label = {i: l for l, i in label2id.items()}

train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": y_train.tolist()})
val_ds   = Dataset.from_dict({"text": X_val.tolist(),   "labels": y_val.tolist()})
test_ds  = Dataset.from_dict({"text": test_df["text"].tolist(), "labels": y_test.tolist()})
print(NUM_LABELS, len(train_ds), len(val_ds), len(test_ds))

In [ ]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = np.array([len(ids) for ids in tokenizer(X_train.tolist())["input_ids"]])

print(pd.Series(lengths).describe(percentiles=[.5, .9, .95, .99]))

MAX_LENGTH = 64
print("Coverage at MAX_LENGTH:", f"{(lengths <= MAX_LENGTH).mean().round(4)}%")

In [ ]:
def tokenize(batch):
  return tokenizer(batch["text"], max_length = MAX_LENGTH, truncation = True)

train_tok = train_ds.map(tokenize, batched = True, remove_columns = ["text"])
val_tok = val_ds.map(tokenize, batched = True, remove_columns = ["text"])
test_tok = test_ds.map(tokenize, batched = True, remove_columns = ["text"])

collector = DataCollatorWithPadding(tokenizer = tokenizer)

In [ ]:
def compute_metrics(eval_pred):
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis= -1)
   return {
       "Accuracy_score":accuracy_score(labels, predictions),
       "F1_score":f1_score(labels, predictions, average="macro")
   }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = NUM_LABELS,
    label2id = label2id,
    id2label= id2label)

In [ ]:
args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "artifacts/transformer/checkpoints"),
    num_train_epochs=8,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps = 25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="F1_score",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collector,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result.metrics)

In [ ]:
val_metrics = trainer.evaluate(val_tok)
print(val_metrics)

In [ ]:
pred_out  = trainer.predict(val_tok)
val_preds = np.argmax(pred_out.predictions, axis=-1)
val_true = pred_out.label_ids

In [ ]:
names = [id2label[i] for i in range(NUM_LABELS)]

report = classification_report(
    val_true,
    val_preds,
    target_names = names,
    labels = list(range(NUM_LABELS)),
    output_dict= True, zero_division=0)
per_class = pd.DataFrame(report).T.iloc[:NUM_LABELS].sort_values("f1-score")

pairs = Counter((id2label[t], id2label[p]) for t, p in zip(val_true, val_preds) if t != p )

for (t, p), n in pairs.most_common(20):
  print(f"{n}  {t}  -> {p}")

In [ ]:
errors = pd.DataFrame({
    "text":val_ds["text"],
    "True":[id2label[i] for i in val_true],
    "Pred":[id2label[i] for i in val_preds]
})
errors = errors[errors["True"] != errors["Pred"]]



In [ ]:
print(f"{len(errors)} errors out of {len(val_ds)}")
print(len(errors))

In [ ]:
print(errors.sample(min(20, len(errors)), random_state = SEED).to_string())

In [ ]:
test_evaluation = trainer.evaluate(test_tok)
pred_out = trainer.predict(test_tok)
test_preds = np.argmax(pred_out.predictions, axis= -1)
test_true = pred_out.label_ids

# Classification Report

In [ ]:
report = classification_report(
    test_true,
    test_preds,
    target_names = [id2label[i] for i in range(NUM_LABELS)],
    output_dict =True,
    zero_division = 0,
    labels= list(range(NUM_LABELS))
)

report_df = pd.DataFrame(report).T.iloc[:NUM_LABELS].sort_values("f1-score")
print(report_df)

In [ ]:
pairs = Counter((id2label[t], id2label[p]) for t, p in zip(test_true, test_preds) if t != p)

for (t, p) , n in pairs.most_common(20):
  print(f"{n:>3}  {t} ---> {p}")

In [ ]:
MODEL_DIR = PROJECT_ROOT / "artifacts/transformer/distilbert-banking77-best"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_DIR))

clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1)
print(clf("My card hasn't arrived yet, when will it get here?", top_k=3))

In [ ]:
results = {
    "model": MODEL_NAME,
    "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
    "train_runtime_sec": round(train_result.metrics["train_runtime"], 1),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "val_accuracy": val_metrics["eval_Accuracy_score"],
    "val_macro_f1": val_metrics["eval_F1_score"],
    "test_accuracy": test_evaluation["eval_Accuracy_score"],
    "test_macro_f1": test_evaluation["eval_F1_score"],
}
print(json.dumps(results, indent=2))

RESULTS_PATH = PROJECT_ROOT / "artifacts/transformer/distilbert_banking77_results.json"

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
print(test_evaluation)

In [79]:
MODEL_DIR = PROJECT_ROOT / "artifacts" / "transformer"/ "distilbert-banking77-best"

load_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
load_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)


clf = pipeline(
    task="text-classification",
    model= load_model,
    tokenizer = load_tokenizer,
    device= 0 if torch.cuda.is_available() else -1)


prediction = clf(
    "My card hasn't arrived yet, when will it get here?",
    top_k=3,
)
print(prediction)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'card_arrival', 'score': 0.9331218600273132}, {'label': 'card_delivery_estimate', 'score': 0.06122898682951927}, {'label': 'transfer_not_received_by_recipient', 'score': 0.0008408245630562305}]


In [ ]:
with open(RESULTS_PATH, "r") as f :
    distilbert_results = json.load(f)

for key, value in distilbert_results.items():
    print(key, value)

model distilbert/distilbert-base-uncased
params_millions 67.0
train_runtime_sec 148.9
best_checkpoint /Workspace/Projects/NLP/banking77-intent-classification/artifacts/transformer/checkpoints/checkpoint-2008
val_accuracy 0.9170414792603698
val_macro_f1 0.9183210286300239
test_accuracy 0.9204545454545454
test_macro_f1 0.9203083016225829


In [ ]:
NEURAL_PATH = PROJECT_ROOT / "reports"/"neural"/ "neural_model_comparison.csv"
neural_df = pd.read_csv(NEURAL_PATH)

TRANSFORMER_PATH = PROJECT_ROOT / "artifacts" / "transformer" / "distilbert_banking77_results.json"
with open(TRANSFORMER_PATH, "r") as f:
    transformer_result = json.load(f)

In [103]:
print("NEURAL MODEL RESULTS")
print(neural_df.to_string(index=False))
print("\nEXISTING TRANSFORMER RESULTS")
print(json.dumps(transformer_result, indent= 2))

NEURAL MODEL RESULTS
            model  parameters  best_epoch  epochs_completed  training_time_seconds  val_loss  val_accuracy  val_macro_f1  val_weighted_f1
embedding_bag_mlp      323789          23                28                   14.3  0.483560         0.872      0.871816         0.871326
              gru      515277           8                13                   14.5  0.499388         0.861      0.859732         0.860576
             lstm      581325          10                15                   18.5  0.517033         0.851      0.851200         0.850188
              rnn      383181          14                19                   18.3  0.726880         0.806      0.803618         0.804233

EXISTING TRANSFORMER RESULTS
{
  "model": "distilbert/distilbert-base-uncased",
  "params_millions": 67.0,
  "train_runtime_sec": 148.9,
  "best_checkpoint": "/Workspace/Projects/NLP/banking77-intent-classification/artifacts/transformer/checkpoints/checkpoint-2008",
  "val_accuracy": 0.9

In [122]:
distilbert_row = pd.DataFrame([{
    "model": "distilbert",
    "parameters": 67_000_000,
    "training_time_seconds": 148.9,
    "val_accuracy": 0.9170414792603698,
    "val_macro_f1": 0.9183210286300239,
}])

neural_comparison = neural_df[[
        
        "model",
        "parameters",
        "training_time_seconds",
        "val_accuracy",
        "val_macro_f1",
]].copy()

comparison_df = pd.concat([distilbert_row, neural_comparison], ignore_index= True)
comparison_df.sort_values(by = "val_macro_f1", ascending=False).reset_index(drop=True)
print(comparison_df.to_string(index=False))

            model  parameters  training_time_seconds  val_accuracy  val_macro_f1
       distilbert    67000000                  148.9      0.917041      0.918321
embedding_bag_mlp      323789                   14.3      0.872000      0.871816
              gru      515277                   14.5      0.861000      0.859732
             lstm      581325                   18.5      0.851000      0.851200
              rnn      383181                   18.3      0.806000      0.803618


In [124]:
REPORT_DIR = PROJECT_ROOT / "reports" / "model_comparison"
REPORT_DIR.mkdir(parents=True, exist_ok = True)

comparison_df.to_csv(REPORT_DIR / "validation_model_comparison.csv",
                      index = False)

comparison_df.to_json(REPORT_DIR / "validation_model_comparison.json",
                      orient = "records",
                      indent = 2)

In [ ]:
class ArtificialModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(10, 8)
        self.layer2 = nn.Linear(8, 4)
        self.output = nn.Linear(4, 2)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        return self.output(x)


toy_model = ArtificialModel()

In [ ]:
for name, param in toy_model.named_parameters():
    print(name, param.shape, param.numel())